# Part E · The written justification (15 marks)

One page. The numbers in this page are **computed live** from the data, so they cannot drift from Part D.
**Run order:** *Run All*; then export the rendered page (see README) and trim it to one page.
Cells 1-3 recompute the audit trail; the page itself is the last Markdown block.

In [1]:
# --- Setup: finds vizlib.py and data/ wherever this notebook is saved ---------------
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore", category=FutureWarning)

def find_assignments_dir():
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        for cand in (p, p / "assignments", p / "week3" / "assignments"):
            if (cand / "vizlib.py").exists() and (cand / "data").is_dir():
                return cand
    raise FileNotFoundError(
        "Could not find vizlib.py + data/. Save this notebook inside "
        "week3/assignments/ (next to vizlib.py) - see README.md.")

ROOT = Path(r"C:\Users\Nas Graphy\Downloads\Data-Visualization\Week 3")
sys.path.insert(0, str(ROOT))
DATA = ROOT / "data"
OUT = ROOT / "outputs"            # every PNG this notebook writes lands here
OUT.mkdir(exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
from vizlib import (PALETTE, MARKERS, style_defaults, save_fig, lie_factor,
                    lie_verdict, simulate_cvd, worst_pair, verdict, delta_e)

style_defaults()                  # house look, set once (slide 26)
BLUE, ORANGE, GREEN = PALETTE["series"]
print("ROOT:", ROOT)
from scipy import stats

WARD_MAP = {"general medicine": "General Medicine", "gen medicine": "General Medicine",
            "cardiology": "Cardiology", "cardio": "Cardiology",
            "day surgery": "Day Surgery", "day-surgery": "Day Surgery",
            "intensive care": "ICU", "icu": "ICU", "i.c.u": "ICU"}
HR_LO, HR_HI = 30, 250

raw = pd.read_csv(DATA / "vitals_10k.csv")
df = raw.copy()
df["ward_clean"] = df.ward.str.strip().str.lower().map(WARD_MAP)
df["bmi"] = df.bmi.replace({999: np.nan, 0: np.nan})
df["heart_rate_raw"] = df.heart_rate
df["heart_rate"] = df.heart_rate.where(df.heart_rate.between(HR_LO, HR_HI))
print("recomputed the same cleaning rules as Part D")

ROOT: C:\Users\Nas Graphy\Downloads\Data-Visualization\Week 3
recomputed the same cleaning rules as Part D


## The numbers behind item 3 (every value removed)

In [2]:
n = {}
n["wards_raw"], n["wards_clean"] = raw.ward.nunique(), df.ward_clean.nunique()
n["bmi999"], n["bmi0"] = int((raw.bmi == 999).sum()), int((raw.bmi == 0).sum())
n["bmi_before"], n["bmi_after"] = raw.bmi.mean(), df.bmi.mean()
n["hr_hi"] = int((raw.heart_rate > HR_HI).sum())
n["hr_lo"] = int((raw.heart_rate < HR_LO).sum())
n["hr_before"], n["hr_after"] = raw.heart_rate.mean(), df.heart_rate.mean()
n["hr_max"] = int(raw.heart_rate.max())
n["zero_los"] = int((raw.los_days == 0).sum())

# did the heart-rate exclusion change any conclusion?  (symmetry test, in numbers)
cols = ["age", "systolic", "diastolic", "heart_rate", "spo2", "bmi", "chol_mmol_l", "glucose_mmol_l", "los_days"]
with_excl = df[cols].corr()
no_excl = df.assign(heart_rate=df.heart_rate_raw)[cols].corr()
n["max_corr_shift"] = float((with_excl.loc["heart_rate"] - no_excl.loc["heart_rate"]).drop("heart_rate").abs().max())
n["max_abs_hr_corr_with"] = float(with_excl.loc["heart_rate"].drop("heart_rate").abs().max())
n["max_abs_hr_corr_without"] = float(no_excl.loc["heart_rate"].drop("heart_rate").abs().max())
n["r_age_chol"] = float(df.age.corr(df.chol_mmol_l))
n["r_sys_dia"] = float(df.systolic.corr(df.diastolic))
pd.Series(n).round(3)

wards_raw                   12.000
wards_clean                  4.000
bmi999                     402.000
bmi0                        84.000
bmi_before                  66.177
bmi_after                   27.346
hr_hi                       29.000
hr_lo                        8.000
hr_before                   78.833
hr_after                    78.047
hr_max                     415.000
zero_los                     2.000
max_corr_shift               0.021
max_abs_hr_corr_with         0.015
max_abs_hr_corr_without      0.016
r_age_chol                   0.340
r_sys_dia                    0.888
dtype: float64

## The one-page justification

In [3]:
page = f"""
### Week 3 · Part E · Justification for the clinical review figure

**1 · Per panel: the question, the chart, the channel rule (Week 2).**

| Panel | Question | Why this chart | Channel rule followed |
|---|---|---|---|
| 1 Histogram of systolic BP | Is systolic pressure one population? | A histogram shows shape; a mean and SD would hide two humps. | Quantity read by **length from a zero baseline**, one hue; the two modes are marked with lines **and** printed numbers. |
| 2 Box plot of stay by ward | Do wards differ in length of stay? | A box plot compares medians and spread across categories on one aligned axis. | **Position on a common scale**; wards **sorted by median** so the order is the ranking; one hue, because colour would add nothing. |
| 3 Scatter, age vs cholesterol | Does cholesterol rise with age? | Two continuous variables, so position on x and y. Panel 4 chose this pair. | **Position** carries both variables; alpha handles overplotting; the trend line is the only highlight colour. |
| 4 Correlation heatmap | Which indicators move together? | Screening 36 pairs at once; this is where panel 3's pair comes from. | Colour is the *weakest* channel, so every cell **also prints its number**; a correlation has a sign, so a diverging map with a neutral midpoint. |

**2 · Every parameter I chose, and why.**
Panel 1: bin width **4 mmHg** (Freedman-Diaconis suggests 3.8; 4 aligns to whole-mmHg readings, and 3 bins hid the humps, 400 bins left 296 empty). Panel 2: **log scale** (stays are right-skewed to {raw.los_days.max():.0f} days, so a linear axis collapses the boxes), `showfliers=True`, default 1.5xIQR whiskers computed on the log scale. Panel 3: **alpha 0.12**, all {len(df):,} points, OLS line (no subsample). Panel 4: **Pearson**, pairwise-complete, `vmin=-1, vmax=+1, center=0`, `RdBu_r`, upper triangle masked, `fmt=".2f"`; `map_mmhg` excluded because it is computed from systolic and diastolic. Export: 300 dpi, `bbox_inches="tight"`, course colour-blind-safe palette (3 slots).

**3 · Every value I removed: rule, count, before and after.** No rows were deleted; invalid *values* were set to missing so each patient still counts for every variable that is valid.

| Rule | Count | Effect |
|---|---|---|
| `bmi` sentinels 999 and 0 are "missing", not measurements | **{n["bmi999"] + n["bmi0"]}** ({n["bmi999"]} x 999, {n["bmi0"]} x 0) | mean BMI **{n["bmi_before"]:.2f} to {n["bmi_after"]:.2f}** kg/m2 |
| `heart_rate` above {HR_HI} bpm is a device fault | **{n["hr_hi"]}** (max {n["hr_max"]} bpm) | see ethics statement |
| `heart_rate` below {HR_LO} bpm (reads 0, monitor off) is a device fault | **{n["hr_lo"]}** | see ethics statement |
| `ward`: {n["wards_raw"]} spellings conformed to {n["wards_clean"]} | 0 values lost | every group-by is now correct |
| `los_days` = 0 cannot be drawn on a log axis | **{n["zero_los"]}** (panel 2 only, declared on the panel) | medians unchanged |

**Ethics statement.** Readings above {HR_HI} bpm were removed as device faults (n = {n["hr_hi"]}), as were {n["hr_lo"]} readings below {HR_LO} bpm. Mean heart rate moves from **{n["hr_before"]:.2f}** to **{n["hr_after"]:.2f}** bpm. The figure was also produced without the exclusion; it is in the appendix (`outputs/partD_figure_appendix_no_hr_exclusion.png`).

**Symmetry test.** Would I have made the same removal if it had *weakened* my conclusion? Yes. The range ({HR_LO}-{HR_HI} bpm) is physiological, fixed before I looked at any correlation, and applied to every row alike. It also barely matters: the largest change to any heart-rate correlation is **{n["max_corr_shift"]:.3f}**, and heart rate correlates with nothing either way (largest |r| {n["max_abs_hr_corr_without"]:.2f} raw vs {n["max_abs_hr_corr_with"]:.2f} cleaned). The conclusions of the figure (two BP populations, age-cholesterol r = {n["r_age_chol"]:.2f}) do not depend on it. That is a rule, not an edit.

*Source: `vitals_10k.csv`, 10,000 synthetic records generated for CDB601220.*
"""
display(Markdown(page))
print("words:", len(page.split()))


### Week 3 · Part E · Justification for the clinical review figure

**1 · Per panel: the question, the chart, the channel rule (Week 2).**

| Panel | Question | Why this chart | Channel rule followed |
|---|---|---|---|
| 1 Histogram of systolic BP | Is systolic pressure one population? | A histogram shows shape; a mean and SD would hide two humps. | Quantity read by **length from a zero baseline**, one hue; the two modes are marked with lines **and** printed numbers. |
| 2 Box plot of stay by ward | Do wards differ in length of stay? | A box plot compares medians and spread across categories on one aligned axis. | **Position on a common scale**; wards **sorted by median** so the order is the ranking; one hue, because colour would add nothing. |
| 3 Scatter, age vs cholesterol | Does cholesterol rise with age? | Two continuous variables, so position on x and y. Panel 4 chose this pair. | **Position** carries both variables; alpha handles overplotting; the trend line is the only highlight colour. |
| 4 Correlation heatmap | Which indicators move together? | Screening 36 pairs at once; this is where panel 3's pair comes from. | Colour is the *weakest* channel, so every cell **also prints its number**; a correlation has a sign, so a diverging map with a neutral midpoint. |

**2 · Every parameter I chose, and why.**
Panel 1: bin width **4 mmHg** (Freedman-Diaconis suggests 3.8; 4 aligns to whole-mmHg readings, and 3 bins hid the humps, 400 bins left 296 empty). Panel 2: **log scale** (stays are right-skewed to 274 days, so a linear axis collapses the boxes), `showfliers=True`, default 1.5xIQR whiskers computed on the log scale. Panel 3: **alpha 0.12**, all 10,000 points, OLS line (no subsample). Panel 4: **Pearson**, pairwise-complete, `vmin=-1, vmax=+1, center=0`, `RdBu_r`, upper triangle masked, `fmt=".2f"`; `map_mmhg` excluded because it is computed from systolic and diastolic. Export: 300 dpi, `bbox_inches="tight"`, course colour-blind-safe palette (3 slots).

**3 · Every value I removed: rule, count, before and after.** No rows were deleted; invalid *values* were set to missing so each patient still counts for every variable that is valid.

| Rule | Count | Effect |
|---|---|---|
| `bmi` sentinels 999 and 0 are "missing", not measurements | **486** (402 x 999, 84 x 0) | mean BMI **66.18 to 27.35** kg/m2 |
| `heart_rate` above 250 bpm is a device fault | **29** (max 415 bpm) | see ethics statement |
| `heart_rate` below 30 bpm (reads 0, monitor off) is a device fault | **8** | see ethics statement |
| `ward`: 12 spellings conformed to 4 | 0 values lost | every group-by is now correct |
| `los_days` = 0 cannot be drawn on a log axis | **2** (panel 2 only, declared on the panel) | medians unchanged |

**Ethics statement.** Readings above 250 bpm were removed as device faults (n = 29), as were 8 readings below 30 bpm. Mean heart rate moves from **78.83** to **78.05** bpm. The figure was also produced without the exclusion; it is in the appendix (`outputs/partD_figure_appendix_no_hr_exclusion.png`).

**Symmetry test.** Would I have made the same removal if it had *weakened* my conclusion? Yes. The range (30-250 bpm) is physiological, fixed before I looked at any correlation, and applied to every row alike. It also barely matters: the largest change to any heart-rate correlation is **0.021**, and heart rate correlates with nothing either way (largest |r| 0.02 raw vs 0.01 cleaned). The conclusions of the figure (two BP populations, age-cholesterol r = 0.34) do not depend on it. That is a rule, not an edit.

*Source: `vitals_10k.csv`, 10,000 synthetic records generated for CDB601220.*


words: 622
